In [ ]:
# !pip install transformers datasets peft trl bitsandbytes accelerate
# from huggingface_hub import notebook_login
# notebook_login()

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# 1. Configuration Constants
MODEL_ID = "meta-llama/Llama-3-8B"
DATASET_ID = "timdettmers/openassistant-guanaco"

# 2. BitsAndBytes 4-bit Quantization Config (Saves massive VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 3. Load Tokenizer and Quantized Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto" # Automatically offloads layers across available GPUs
)

# Prepare model weights for gradient checkpointing/kbit training
model = prepare_model_for_kbit_training(model)

# 4. PEFT (LoRA) Configuration
peft_config = LoraConfig(
    r=16,               # Rank of update matrices
    lora_alpha=32,      # Scaling factor
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # Target attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# 5. Load Training Dataset
dataset = load_dataset(DATASET_ID, split="train[:1000]") # Subset for demo

# 6. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100,
    fp16=True,
    optim="paged_adamw_8bit" # Optimizes VRAM usage during backprop
)

# 7. Initialize SFTTrainer and Start Fine-Tuning
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

print("\n--- Starting Fine-Tuning ---")
trainer.train()

# 8. Save the trained LoRA adapters
trainer.model.save_pretrained("./fine_tuned_adapters")